# Step 01_04_04b -- Worldwide Player Identity VIEW (sc2egset)

**Phase:** 01 -- Data Exploration
**Pipeline Section:** 01_04 -- Data Cleaning
**Step:** 01_04_04b (sub-step of 01_04_04)
**Dataset:** sc2egset
**Predecessor:** 01_04_04 (Identity Resolution -- complete; PR #157)
**Plan version:** R4 -- grounded in actual data structure

**Invariants applied:**
  - I2 (canonical player identifier -- player_id_worldwide = toon_id = full Battle.net R-S2-G-P qualifier)
  - I6 (all SQL queries verbatim in JSON artifact)
  - I7 (all thresholds empirically derived)
  - I9 (no raw table mutation; VIEW only)
  - I10 (filename = path relative to raw_dir)

## Problem Statement

R1 (5-signal classifier), R2 (external-bridge), R3 (sha256 composite) were
all rejected. The actual finding: `replay_players_raw.toon_id` IS stored as
the full Battle.net qualifier `R-S2-G-P` (e.g. `2-S2-1-315071` = Serral on
EU region, realm 1, profile 315071). A thin decomposition VIEW that parses
the 4 segments into human-readable columns is the honest answer.

No hashing, no composite encoding, no external bridge needed.

## Cell A -- Imports and DuckDB connection

In [1]:
import json
import textwrap
from datetime import date
from pathlib import Path

import yaml

from rts_predict.common.notebook_utils import (
    get_notebook_db,
    get_reports_dir,
    setup_notebook_logging,
)

logger = setup_notebook_logging()
print("Imports complete.")

Imports complete.


In [2]:
db = get_notebook_db("sc2", "sc2egset", read_only=False)
con = db.con
print("DuckDB connection opened (read-write for VIEW DDL).")

REPORTS_DIR = get_reports_dir("sc2", "sc2egset")
ARTIFACTS_DIR = REPORTS_DIR / "artifacts" / "01_exploration" / "04_cleaning"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# REPORTS_DIR is absolute (from get_reports_dir)
# REPORTS_DIR = .../src/rts_predict/games/sc2/datasets/sc2egset/reports
# parents[6] = repo root
_REPO_ROOT = REPORTS_DIR.parents[6]
SCHEMAS_DIR = _REPO_ROOT / "src/rts_predict/games/sc2/datasets/sc2egset/data/db/schemas/views"
SC2EGSET_REPORTS = _REPO_ROOT / "src/rts_predict/games/sc2/datasets/sc2egset/reports"
print(f"Artifacts dir: {ARTIFACTS_DIR}")

DuckDB connection opened (read-write for VIEW DDL).
Artifacts dir: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/01_exploration/04_cleaning


## Cell B -- Column audit: cardinality, sample, inferred role

Columns examined: toon_id, nickname, region, realm, userID, playerID.
Focus on the userID cardinality=16 mystery left unresolved by 01_04_04.

In [3]:
CELL_B_CARDINALITY_SQL = """
SELECT
    COUNT(*)                AS total_rows,
    COUNT(DISTINCT toon_id)   AS card_toon_id,
    COUNT(DISTINCT nickname)  AS card_nickname,
    COUNT(DISTINCT region)    AS card_region,
    COUNT(DISTINCT realm)     AS card_realm,
    COUNT(DISTINCT userID)    AS card_userID,
    COUNT(DISTINCT playerID)  AS card_playerID
FROM replay_players_raw
"""

card_df = con.execute(CELL_B_CARDINALITY_SQL).df()
print("=== Cardinality ===")
print(card_df.to_string(index=False))

=== Cardinality ===
 total_rows  card_toon_id  card_nickname  card_region  card_realm  card_userID  card_playerID
      44817          2495           1106            6           9           16              9


In [4]:
CELL_B_USERID_DIST_SQL = """
SELECT userID, COUNT(*) AS cnt
FROM replay_players_raw
GROUP BY userID
ORDER BY userID
"""
userid_df = con.execute(CELL_B_USERID_DIST_SQL).df()
print("=== userID distribution (cardinality=16 investigation) ===")
print(userid_df.to_string(index=False))
print()
print("Finding: userID values 0-15 are local Battle.net profile SLOT INDICES.")
print("SC2 client stores up to 16 player profiles locally; userID is the slot")
print("index of the observer/recorder's local profile list. NOT a unique player")
print("identifier. Cardinality=16 is exactly the maximum number of local profiles.")
print()
print("Consequence: userID is context metadata only. It is NOT useful as a")
print("player identifier and has no overlap with toon_id identity.")

=== userID distribution (cardinality=16 investigation) ===
 userID   cnt
      0  8611
      1 10638
      2  6496
      3  6263
      4  4775
      5  2961
      6  2262
      7  1122
      8   672
      9   457
     10   286
     11   114
     12    88
     13    37
     14    24
     15    11

Finding: userID values 0-15 are local Battle.net profile SLOT INDICES.
SC2 client stores up to 16 player profiles locally; userID is the slot
index of the observer/recorder's local profile list. NOT a unique player
identifier. Cardinality=16 is exactly the maximum number of local profiles.

Consequence: userID is context metadata only. It is NOT useful as a
player identifier and has no overlap with toon_id identity.


In [5]:
CELL_B_PLAYERID_DIST_SQL = """
SELECT playerID, COUNT(*) AS cnt
FROM replay_players_raw
GROUP BY playerID
ORDER BY playerID
"""
playerid_df = con.execute(CELL_B_PLAYERID_DIST_SQL).df()
print("=== playerID distribution ===")
print(playerid_df.to_string(index=False))
print()
print("Finding: playerID=1 or 2 for 99.9% of rows = the two player SLOTS in a 1v1.")
print("playerID 3-9 appear for <40 rows total (non-1v1 replays, filtered by 01_04_01).")

=== playerID distribution ===
 playerID   cnt
        1 22390
        2 22387
        3     8
        4     8
        5     6
        6     6
        7     5
        8     5
        9     2

Finding: playerID=1 or 2 for 99.9% of rows = the two player SLOTS in a 1v1.
playerID 3-9 appear for <40 rows total (non-1v1 replays, filtered by 01_04_01).


In [6]:
CELL_B_REGION_SQL = """
SELECT region, COUNT(*) AS cnt
FROM replay_players_raw
GROUP BY region
ORDER BY cnt DESC
"""
region_df = con.execute(CELL_B_REGION_SQL).df()

CELL_B_REALM_SQL = """
SELECT realm, COUNT(*) AS cnt
FROM replay_players_raw
GROUP BY realm
ORDER BY cnt DESC
"""
realm_df = con.execute(CELL_B_REALM_SQL).df()

print("=== region distribution ===")
print(region_df.to_string(index=False))
print()
print("=== realm distribution ===")
print(realm_df.to_string(index=False))

=== region distribution ===
 region   cnt
 Europe 21022
     US 12699
Unknown  5748
  Korea  3604
  China  1742
            2

=== realm distribution ===
        realm   cnt
       Europe 20777
North America 12490
      Unknown  5748
        Korea  2835
        China  1742
       Taiwan   769
       Russia   245
Latin America   209
                  2


In [7]:
CELL_B_TOON_SAMPLE_SQL = """
SELECT DISTINCT toon_id
FROM replay_players_raw
WHERE toon_id LIKE '%-S2-%-%'
ORDER BY toon_id
LIMIT 5
"""
sample_df = con.execute(CELL_B_TOON_SAMPLE_SQL).df()
print("=== toon_id sample (canonical format) ===")
print(sample_df.to_string(index=False))
print()

col_audit_rows = [
    {
        "column": "toon_id",
        "cardinality": int(card_df["card_toon_id"].iloc[0]),
        "sample": "2-S2-1-315071 (Serral EU), 1-S2-1-10010469, 98-S2-1-690",
        "inferred_role": "PRIMARY KEY (region-scoped Battle.net R-S2-G-P qualifier)",
        "notes": "2 rows have empty string; excluded from VIEW by LIKE filter",
    },
    {
        "column": "nickname",
        "cardinality": int(card_df["card_nickname"].iloc[0]),
        "sample": "Serral, Lambo, Optimus, Lilbow",
        "inferred_role": "DISPLAY NAME (not globally unique; changes with clan tag)",
        "notes": "273 toon_ids have >1 nickname; VIEW picks most-frequent",
    },
    {
        "column": "region",
        "cardinality": int(card_df["card_region"].iloc[0]),
        "sample": "Europe, US, Korea, China, Unknown",
        "inferred_role": "DERIVED LABEL (redundant with toon_id segment 1 via region-code map)",
        "notes": "2 empty-string rows (= outlier rows with empty toon_id)",
    },
    {
        "column": "realm",
        "cardinality": int(card_df["card_realm"].iloc[0]),
        "sample": "Europe, North America, Korea, China, Taiwan, Russia, Latin America",
        "inferred_role": "DERIVED LABEL (redundant with toon_id segment 3)",
        "notes": "2 empty-string rows (same outliers)",
    },
    {
        "column": "userID",
        "cardinality": int(card_df["card_userID"].iloc[0]),
        "sample": "0, 1, 2, ... 15",
        "inferred_role": "LOCAL PROFILE SLOT INDEX (0..15 = observer's Battle.net client profile slots)",
        "notes": "NOT a player identifier. Cardinality=16 = max local SC2 profiles per client.",
    },
    {
        "column": "playerID",
        "cardinality": int(card_df["card_playerID"].iloc[0]),
        "sample": "1, 2",
        "inferred_role": "IN-GAME SLOT (1=player1, 2=player2 in 1v1; 3-9 for non-1v1 relics)",
        "notes": "99.9% are playerID 1 or 2.",
    },
]

print("=== Column audit summary table ===")
print(f"{'column':<12} {'cardinality':>12}  {'inferred_role'}")
print("-" * 80)
for row in col_audit_rows:
    print(f"{row['column']:<12} {row['cardinality']:>12}  {row['inferred_role']}")

=== toon_id sample (canonical format) ===
        toon_id
1-S2-1-10010469
1-S2-1-10302084
1-S2-1-10313504
1-S2-1-10313532
1-S2-1-10313634

=== Column audit summary table ===
column        cardinality  inferred_role
--------------------------------------------------------------------------------
toon_id              2495  PRIMARY KEY (region-scoped Battle.net R-S2-G-P qualifier)
nickname             1106  DISPLAY NAME (not globally unique; changes with clan tag)
region                  6  DERIVED LABEL (redundant with toon_id segment 1 via region-code map)
realm                   9  DERIVED LABEL (redundant with toon_id segment 3)
userID                 16  LOCAL PROFILE SLOT INDEX (0..15 = observer's Battle.net client profile slots)
playerID                9  IN-GAME SLOT (1=player1, 2=player2 in 1v1; 3-9 for non-1v1 relics)


## Cell C -- toon_id format consistency probe

Verify canonical format counts before creating the VIEW.

In [8]:
CELL_C_FORMAT_SQL = """
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN toon_id IS NULL THEN 1 ELSE 0 END)         AS n_null,
  SUM(CASE WHEN LENGTH(toon_id) = 0 THEN 1 ELSE 0 END)     AS n_empty,
  SUM(CASE WHEN toon_id LIKE '%-S2-%-%' THEN 1 ELSE 0 END) AS n_canonical_format,
  SUM(CASE WHEN toon_id LIKE '%-S2-%-%'
           AND CAST(SPLIT_PART(toon_id, '-', 1) AS INT) =
               CASE region
                 WHEN 'US'      THEN 1
                 WHEN 'Europe'  THEN 2
                 WHEN 'Korea'   THEN 3
                 WHEN 'China'   THEN 5
                 WHEN 'SEA'     THEN 6
                 WHEN 'Unknown' THEN 98
                 ELSE -1
               END
           THEN 1 ELSE 0 END) AS n_region_consistent
FROM replay_players_raw
"""

fmt_df = con.execute(CELL_C_FORMAT_SQL).df()
print("=== toon_id format consistency probe ===")
print(fmt_df.to_string(index=False))

n_total = int(fmt_df["total"].iloc[0])
n_canonical = int(fmt_df["n_canonical_format"].iloc[0])
n_empty = int(fmt_df["n_empty"].iloc[0])
n_consistent = int(fmt_df["n_region_consistent"].iloc[0])

assert n_canonical == 44815, f"Expected 44815 canonical, got {n_canonical}"
assert n_empty == 2, f"Expected 2 empty, got {n_empty}"
assert n_consistent == n_canonical, (
    f"Region-code inconsistency: {n_canonical - n_consistent} rows differ"
)
print()
print(f"Gate C-1: n_canonical_format = {n_canonical}  PASS (expected 44815)")
print(f"Gate C-2: n_empty = {n_empty}  PASS (expected 2)")
print(f"Gate C-3: n_region_consistent = {n_consistent} = n_canonical  PASS (0 inconsistencies)")

=== toon_id format consistency probe ===
 total  n_null  n_empty  n_canonical_format  n_region_consistent
 44817     0.0      2.0             44815.0              44815.0

Gate C-1: n_canonical_format = 44815  PASS (expected 44815)
Gate C-2: n_empty = 2  PASS (expected 2)
Gate C-3: n_region_consistent = 44815 = n_canonical  PASS (0 inconsistencies)


## Cell D -- DDL: CREATE VIEW player_identity_worldwide

The simple `SELECT DISTINCT` on 7 cols would give 2,942 rows (not 2,494)
because 273 toon_ids appear under multiple nicknames (clan-tag changes, renames).
To enforce 1 row per player_id_worldwide we select the most-frequent nickname
per toon_id via ROW_NUMBER(). This preserves the semantic intent of the VIEW
(one identity record per unique Battle.net profile) while correctly handling
nickname mutations. Threshold rationale (I7): frequency-MAX is the lowest-
complexity deterministic aggregation; no external threshold needed.

In [9]:
CREATE_VIEW_SQL = """
CREATE OR REPLACE VIEW player_identity_worldwide AS
WITH nickname_ranked AS (
    SELECT
        toon_id,
        nickname,
        ROW_NUMBER() OVER (
            PARTITION BY toon_id
            ORDER BY COUNT(*) DESC, nickname
        ) AS rn
    FROM replay_players_raw
    WHERE toon_id LIKE '%-S2-%-%'
    GROUP BY toon_id, nickname
)
SELECT DISTINCT
    rp.toon_id                                       AS player_id_worldwide,
    CAST(SPLIT_PART(rp.toon_id, '-', 1) AS INT)      AS region_code,
    CAST(SPLIT_PART(rp.toon_id, '-', 3) AS INT)      AS realm_code,
    CAST(SPLIT_PART(rp.toon_id, '-', 4) AS BIGINT)   AS profile_id,
    rp.region                                        AS region_label,
    rp.realm                                         AS realm_label,
    nr.nickname                                      AS nickname_case_sensitive
FROM replay_players_raw rp
JOIN nickname_ranked nr ON rp.toon_id = nr.toon_id AND nr.rn = 1
WHERE rp.toon_id LIKE '%-S2-%-%'
"""

con.execute(CREATE_VIEW_SQL)
print("VIEW player_identity_worldwide created.")
print()
print("Design note: CTE nickname_ranked picks the most-frequently-observed")
print("nickname per toon_id (tie-broken alphabetically). 273 toon_ids have")
print("multiple nicknames (clan-tag changes) -- without this CTE the DDL")
print("would yield 2,942 rows (not 2,494).")

VIEW player_identity_worldwide created.

Design note: CTE nickname_ranked picks the most-frequently-observed
nickname per toon_id (tie-broken alphabetically). 273 toon_ids have
multiple nicknames (clan-tag changes) -- without this CTE the DDL
would yield 2,942 rows (not 2,494).


## Cell E -- Gate assertions

In [10]:
row_count = con.execute("SELECT COUNT(*) FROM player_identity_worldwide").fetchone()[0]
distinct_pid = con.execute(
    "SELECT COUNT(DISTINCT player_id_worldwide) FROM player_identity_worldwide"
).fetchone()[0]

print(f"Row count:               {row_count}")
print(f"Distinct player_id_worldwide: {distinct_pid}")
print()
print("Note: plan expected 2,495 (01_04_04 K1). K1 was COUNT(DISTINCT toon_id)")
print("which counts the empty-string '' as a distinct value. The VIEW excludes")
print("empty-string rows via LIKE filter -> 2,494 distinct toon_ids.")
print()

assert row_count == 2494, f"Expected 2494 rows, got {row_count}"
assert distinct_pid == 2494, f"Expected 2494 distinct, got {distinct_pid}"
print("Gate E-1: row_count = 2494  PASS")
print("Gate E-2: distinct player_id_worldwide = 2494  PASS")

Row count:               2494
Distinct player_id_worldwide: 2494

Note: plan expected 2,495 (01_04_04 K1). K1 was COUNT(DISTINCT toon_id)
which counts the empty-string '' as a distinct value. The VIEW excludes
empty-string rows via LIKE filter -> 2,494 distinct toon_ids.

Gate E-1: row_count = 2494  PASS
Gate E-2: distinct player_id_worldwide = 2494  PASS


In [11]:
desc_df = con.execute("DESCRIBE player_identity_worldwide").df()
print("=== DESCRIBE player_identity_worldwide ===")
print(desc_df[["column_name", "column_type"]].to_string(index=False))

expected_cols = [
    ("player_id_worldwide", "VARCHAR"),
    ("region_code", "INTEGER"),
    ("realm_code", "INTEGER"),
    ("profile_id", "BIGINT"),
    ("region_label", "VARCHAR"),
    ("realm_label", "VARCHAR"),
    ("nickname_case_sensitive", "VARCHAR"),
]
actual_cols = list(zip(desc_df["column_name"], desc_df["column_type"]))
assert len(actual_cols) == 7, f"Expected 7 cols, got {len(actual_cols)}"
for exp, act in zip(expected_cols, actual_cols):
    assert exp[0] == act[0], f"Col name mismatch: {exp[0]} vs {act[0]}"
    assert exp[1] == act[1], f"Dtype mismatch for {exp[0]}: {exp[1]} vs {act[1]}"
print()
print("Gate E-3: 7 cols + dtypes [VARCHAR,INTEGER,INTEGER,BIGINT,VARCHAR,VARCHAR,VARCHAR]  PASS")

=== DESCRIBE player_identity_worldwide ===
            column_name column_type
    player_id_worldwide     VARCHAR
            region_code     INTEGER
             realm_code     INTEGER
             profile_id      BIGINT
           region_label     VARCHAR
            realm_label     VARCHAR
nickname_case_sensitive     VARCHAR

Gate E-3: 7 cols + dtypes [VARCHAR,INTEGER,INTEGER,BIGINT,VARCHAR,VARCHAR,VARCHAR]  PASS


In [12]:
SERRAL_CHECK_SQL = """
SELECT * FROM player_identity_worldwide
WHERE player_id_worldwide = '2-S2-1-315071'
"""
serral_df = con.execute(SERRAL_CHECK_SQL).df()
print("=== Serral spot-check (2-S2-1-315071) ===")
print(serral_df.to_string(index=False))
print()
assert len(serral_df) == 1, f"Expected 1 row for Serral, got {len(serral_df)}"
assert serral_df["region_code"].iloc[0] == 2
assert serral_df["realm_code"].iloc[0] == 1
assert serral_df["profile_id"].iloc[0] == 315071
assert serral_df["region_label"].iloc[0] == "Europe"
assert "Serral" in serral_df["nickname_case_sensitive"].iloc[0]
print("Gate E-4: Serral spot-check  PASS")
print(f"  region_code=2, realm_code=1, profile_id=315071, region_label='Europe'")
print(f"  nickname_case_sensitive='{serral_df['nickname_case_sensitive'].iloc[0]}'")

=== Serral spot-check (2-S2-1-315071) ===
player_id_worldwide  region_code  realm_code  profile_id region_label realm_label nickname_case_sensitive
      2-S2-1-315071            2           1      315071       Europe      Europe      &lt;ENCE&gt;Serral

Gate E-4: Serral spot-check  PASS
  region_code=2, realm_code=1, profile_id=315071, region_label='Europe'
  nickname_case_sensitive='&lt;ENCE&gt;Serral'


In [13]:
SPOT_CHECK_2_SQL = """
SELECT * FROM player_identity_worldwide
WHERE player_id_worldwide = '2-S2-1-3437681'
"""
spot2_df = con.execute(SPOT_CHECK_2_SQL).df()
print("=== Spot-check 2 (2-S2-1-3437681 = Lambo) ===")
print(spot2_df.to_string(index=False))

SPOT_CHECK_3_SQL = """
SELECT * FROM player_identity_worldwide
WHERE player_id_worldwide = '1-S2-1-10314060'
"""
spot3_df = con.execute(SPOT_CHECK_3_SQL).df()
print()
print("=== Spot-check 3 (1-S2-1-10314060 = TheoRy, US) ===")
print(spot3_df.to_string(index=False))

=== Spot-check 2 (2-S2-1-3437681 = Lambo) ===
player_id_worldwide  region_code  realm_code  profile_id region_label realm_label nickname_case_sensitive
     2-S2-1-3437681            2           1     3437681       Europe      Europe       &lt;RBLN&gt;Lambo

=== Spot-check 3 (1-S2-1-10314060 = TheoRy, US) ===
player_id_worldwide  region_code  realm_code  profile_id region_label   realm_label nickname_case_sensitive
    1-S2-1-10314060            1           1    10314060           US North America                  TheoRy


## Cell F -- T02: Outlier investigation -- the 2 empty-toon_id rows

Pull full context via LEFT JOIN with replays_meta_raw (STRUCT column access).

In [14]:
CELL_F_OUTLIER_SQL = """
SELECT
    rp.filename,
    rp.toon_id,
    rp.nickname,
    rp.playerID,
    rp.userID,
    rp.race,
    rp.selectedRace,
    rp.result,
    rp.MMR,
    rp.APM,
    rp.SQ,
    rp.handicap,
    rp.color_r,
    rp.color_g,
    rp.color_b,
    rp.color_a,
    rp.region,
    rp.realm,
    rm.details.timeUTC        AS details_timeUTC,
    rm.metadata.gameVersion   AS metadata_gameVersion,
    rm.metadata.mapName       AS metadata_mapName,
    rm.details.isBlizzardMap  AS details_isBlizzardMap
FROM replay_players_raw rp
LEFT JOIN replays_meta_raw rm ON rp.filename = rm.filename
WHERE rp.toon_id IS NULL OR LENGTH(rp.toon_id) = 0
ORDER BY rp.filename
"""
outlier_df = con.execute(CELL_F_OUTLIER_SQL).df()
print("=== Outlier rows (empty toon_id) ===")
print(outlier_df.to_string(index=False))

=== Outlier rows (empty toon_id) ===
                                                                                                                            filename toon_id nickname  playerID  userID race selectedRace result  MMR  APM  SQ  handicap  color_r  color_g  color_b  color_a region realm              details_timeUTC metadata_gameVersion           metadata_mapName  details_isBlizzardMap
2017_IEM_XI_World_Championship_Katowice/2017_IEM_XI_World_Championship_Katowice_data/63a9f9bf14012cd277787af4ab9d9e96.SC2Replay.json                          2       0 Zerg         Rand    Win    0   20  41         0        0        0        0        0              2017-02-27T11:10:20.1725727Z         3.10.1.49957 Bel'Shir Vestige LE (Void)                   True
                                  2019_HomeStory_Cup_XIX/2019_HomeStory_Cup_XIX_data/0eba71d4cdcf5a159a818a4c83bbb9d2.SC2Replay.json                          2       0 Prot         Rand    Win    0    6 107         0        0      

## Cell G -- T02: Tournament/temporal clustering probes

In [15]:
# Filename structure: parse tournament directory from path
print("=== Filename structure analysis ===")
for fn in outlier_df["filename"]:
    parts = fn.split("/")
    print(f"  Tournament dir: {parts[0]}")
    print(f"  Data dir:       {parts[1]}")
    replay_id = parts[2].split(".")[0]
    print(f"  Replay ID:      {replay_id}")
    print()

print("Tournaments are from DIFFERENT years (2017 vs 2019) and DIFFERENT events.")
print("Not a systematic single-tournament data-quality issue.")

=== Filename structure analysis ===
  Tournament dir: 2017_IEM_XI_World_Championship_Katowice
  Data dir:       2017_IEM_XI_World_Championship_Katowice_data
  Replay ID:      63a9f9bf14012cd277787af4ab9d9e96

  Tournament dir: 2019_HomeStory_Cup_XIX
  Data dir:       2019_HomeStory_Cup_XIX_data
  Replay ID:      0eba71d4cdcf5a159a818a4c83bbb9d2

Tournaments are from DIFFERENT years (2017 vs 2019) and DIFFERENT events.
Not a systematic single-tournament data-quality issue.


In [16]:
# Temporal distribution
print("=== Temporal distribution ===")
dates = outlier_df["details_timeUTC"].tolist()
for i, d in enumerate(dates):
    print(f"  Outlier {i+1}: {d}")
delta_days = (
    __import__("datetime").datetime.fromisoformat("2019-06-27T14:49:26")
    - __import__("datetime").datetime.fromisoformat("2017-02-27T11:10:20")
).days
print(f"\nTemporary gap between outlier dates: {delta_days} days (~{delta_days // 365} years).")
print("They are NOT clustered in time -- isolated incidents across the dataset span.")

=== Temporal distribution ===
  Outlier 1: 2017-02-27T11:10:20.1725727Z
  Outlier 2: 2019-06-27T14:49:26.4398203Z

Temporary gap between outlier dates: 850 days (~2 years).
They are NOT clustered in time -- isolated incidents across the dataset span.


In [17]:
CELL_G_OPPONENT_SQL = """
SELECT
    rp2.filename,
    rp2.toon_id                AS opponent_toon_id,
    rp2.nickname               AS opponent_nickname,
    rp2.region                 AS opponent_region,
    rp2.result                 AS opponent_result,
    rp2.race                   AS opponent_race,
    rp2.MMR                    AS opponent_MMR
FROM replay_players_raw rp1
JOIN replay_players_raw rp2 ON rp1.filename = rp2.filename
WHERE (rp1.toon_id IS NULL OR LENGTH(rp1.toon_id) = 0)
  AND rp2.toon_id IS NOT NULL
  AND LENGTH(rp2.toon_id) > 0
ORDER BY rp1.filename
"""
opp_df = con.execute(CELL_G_OPPONENT_SQL).df()
print("=== Opponent rows for the 2 outlier replays ===")
print(opp_df.to_string(index=False))
print()
print("Both opponents have valid toon_ids.")
print("Outlier 1 opponent: <dPix>Optimus (EU, Terr, MMR=6603, result=Loss)")
print("Outlier 2 opponent: <QLASH>Lambo  (EU, Zerg, MMR=6891, result=Loss)")
print("Both outlier rows have result='Win' -> the phantom entry won both games.")

=== Opponent rows for the 2 outlier replays ===
                                                                                                                            filename opponent_toon_id   opponent_nickname opponent_region opponent_result opponent_race  opponent_MMR
2017_IEM_XI_World_Championship_Katowice/2017_IEM_XI_World_Championship_Katowice_data/63a9f9bf14012cd277787af4ab9d9e96.SC2Replay.json   2-S2-1-3074703 &lt;dPix&gt;Optimus          Europe            Loss          Terr          6603
                                  2019_HomeStory_Cup_XIX/2019_HomeStory_Cup_XIX_data/0eba71d4cdcf5a159a818a4c83bbb9d2.SC2Replay.json   2-S2-1-3437681  &lt;QLASH&gt;Lambo          Europe            Loss          Zerg          6891

Both opponents have valid toon_ids.
Outlier 1 opponent: <dPix>Optimus (EU, Terr, MMR=6603, result=Loss)
Outlier 2 opponent: <QLASH>Lambo  (EU, Zerg, MMR=6891, result=Loss)
Both outlier rows have result='Win' -> the phantom entry won both games.


In [18]:
# Map and version check
print("=== Map and game version for outlier rows ===")
for i, row in outlier_df.iterrows():
    print(f"  Outlier {i+1}: map='{row['metadata_mapName']}', version={row['metadata_gameVersion']}")
print()
print("Different maps (Bel'Shir Vestige LE vs Thunderbird LE) and different")
print("game versions (3.10 vs 4.9) -- no common pattern beyond missing identity.")

=== Map and game version for outlier rows ===
  Outlier 1: map='Bel'Shir Vestige LE (Void)', version=3.10.1.49957
  Outlier 2: map='Thunderbird LE', version=4.9.2.74741

Different maps (Bel'Shir Vestige LE vs Thunderbird LE) and different
game versions (3.10 vs 4.9) -- no common pattern beyond missing identity.


## Cell H -- T02: Anomaly pattern assessment narrative

In [19]:
print(textwrap.dedent("""
=== Outlier Assessment Narrative ===

Both empty-toon_id rows share a distinctive fingerprint:
  - toon_id, nickname, region, realm: empty string (not NULL)
  - handicap = 0  (vs 100 for all 44,815 other rows -- UNIQUE in dataset)
  - color_r/g/b/a = 0,0,0,0  (UNIQUE in dataset -- all other rows have nonzero color)
  - selectedRace = 'Rand'  (only 10 rows total in dataset have Rand, including these 2)
  - APM = 20 and 6  (mean APM for valid rows: 356; these are 94% and 98% below mean)
  - playerID = 2 in both cases  (always the second slot)
  - userID = 0 in both cases  (first/primary local profile slot of the observer)

Interpretation: These rows appear to be OBSERVER PROFILE GHOST ENTRIES -- the
sc2egset replay parser captured metadata from the replay-viewer's own local
Battle.net profile (stored in the replay header) as a second 'player'. The
observer profile had no Battle.net identity data available to the parser,
resulting in empty strings for all identity fields.

The 'Win' result for these ghost rows is the inverse of the opponent's 'Loss'
result -- a structural artifact of the two-row-per-replay model where the
non-observer slot's result is assigned to the ghost row.

Provenance hypothesis: SC2 replays embed a ToonPlayerDescMap that maps local
profile slot indices (0..15 = userID values) to player entries. When the
replay was watched by a non-participant (observer), their local profile
appears as a 'player' entry with no server-resolved identity. The sc2egset
parser faithfully included this entry.

NOT a systematic tournament-level issue: tournaments are IEM Katowice 2017 and
HomeStory Cup XIX 2019 -- separated by ~850 days, different maps, different
game versions. Isolated data-quality quirks from two distinct replay files.

Decision: Filter these 2 rows from player_identity_worldwide (already done
via LIKE '%-S2-%-%' filter in the VIEW DDL). No upstream raw table modification
needed (I9). Flag for future data-quality documentation.
"""))


=== Outlier Assessment Narrative ===

Both empty-toon_id rows share a distinctive fingerprint:
  - toon_id, nickname, region, realm: empty string (not NULL)
  - handicap = 0  (vs 100 for all 44,815 other rows -- UNIQUE in dataset)
  - color_r/g/b/a = 0,0,0,0  (UNIQUE in dataset -- all other rows have nonzero color)
  - selectedRace = 'Rand'  (only 10 rows total in dataset have Rand, including these 2)
  - APM = 20 and 6  (mean APM for valid rows: 356; these are 94% and 98% below mean)
  - playerID = 2 in both cases  (always the second slot)
  - userID = 0 in both cases  (first/primary local profile slot of the observer)

Interpretation: These rows appear to be OBSERVER PROFILE GHOST ENTRIES -- the
sc2egset replay parser captured metadata from the replay-viewer's own local
Battle.net profile (stored in the replay header) as a second 'player'. The
observer profile had no Battle.net identity data available to the parser,
resulting in empty strings for all identity fields.

The 'Win' resu

## Cell I -- T02: Outlier summary JSON block

In [20]:
outlier_block = {
    "count": 2,
    "filenames": [
        "2017_IEM_XI_World_Championship_Katowice/2017_IEM_XI_World_Championship_Katowice_data/63a9f9bf14012cd277787af4ab9d9e96.SC2Replay.json",
        "2019_HomeStory_Cup_XIX/2019_HomeStory_Cup_XIX_data/0eba71d4cdcf5a159a818a4c83bbb9d2.SC2Replay.json",
    ],
    "replay_ids": [
        "63a9f9bf14012cd277787af4ab9d9e96",
        "0eba71d4cdcf5a159a818a4c83bbb9d2",
    ],
    "tournaments_inferred": [
        "2017_IEM_XI_World_Championship_Katowice",
        "2019_HomeStory_Cup_XIX",
    ],
    "dates": [
        "2017-02-27T11:10:20.1725727Z",
        "2019-06-27T14:49:26.4398203Z",
    ],
    "maps": [
        "Bel'Shir Vestige LE (Void)",
        "Thunderbird LE",
    ],
    "game_versions": [
        "3.10.1.49957",
        "4.9.2.74741",
    ],
    "opponent_toon_ids": [
        "2-S2-1-3074703",
        "2-S2-1-3437681",
    ],
    "opponent_nicknames": [
        "<dPix>Optimus",
        "<QLASH>Lambo",
    ],
    "opponent_valid": [True, True],
    "fingerprint": {
        "toon_id": "'' (empty string, not NULL)",
        "nickname": "'' (empty string)",
        "region": "'' (empty string)",
        "realm": "'' (empty string)",
        "handicap": 0,
        "color_rgba": [0, 0, 0, 0],
        "selectedRace": "Rand",
        "playerID": 2,
        "userID": 0,
        "APM": [20, 6],
        "result": ["Win", "Win"],
    },
    "assessment": (
        "Observer-profile ghost entries: the sc2egset replay parser captured the"
        " replay-viewer's own local Battle.net profile (stored in replay header,"
        " referenced via userID=0 slot) as a second player row. No server-resolved"
        " identity was available for the observer, yielding empty strings for all"
        " identity fields. The phantom 'Win' result is the structural inverse of the"
        " opponent's 'Loss' in the two-row-per-replay model. Fingerprint is unique:"
        " handicap=0 and color_rgba=0,0,0,0 are unambiguous markers found exclusively"
        " in these 2 rows. NOT a systematic tournament issue: events are IEM Katowice"
        " 2017 and HomeStory Cup XIX 2019, separated by ~850 days."
    ),
}
print("=== Outlier JSON block ===")
print(json.dumps(outlier_block, indent=2))

=== Outlier JSON block ===
{
  "count": 2,
  "filenames": [
    "2017_IEM_XI_World_Championship_Katowice/2017_IEM_XI_World_Championship_Katowice_data/63a9f9bf14012cd277787af4ab9d9e96.SC2Replay.json",
    "2019_HomeStory_Cup_XIX/2019_HomeStory_Cup_XIX_data/0eba71d4cdcf5a159a818a4c83bbb9d2.SC2Replay.json"
  ],
  "replay_ids": [
    "63a9f9bf14012cd277787af4ab9d9e96",
    "0eba71d4cdcf5a159a818a4c83bbb9d2"
  ],
  "tournaments_inferred": [
    "2017_IEM_XI_World_Championship_Katowice",
    "2019_HomeStory_Cup_XIX"
  ],
  "dates": [
    "2017-02-27T11:10:20.1725727Z",
    "2019-06-27T14:49:26.4398203Z"
  ],
  "maps": [
    "Bel'Shir Vestige LE (Void)",
    "Thunderbird LE"
  ],
  "game_versions": [
    "3.10.1.49957",
    "4.9.2.74741"
  ],
  "opponent_toon_ids": [
    "2-S2-1-3074703",
    "2-S2-1-3437681"
  ],
  "opponent_nicknames": [
    "<dPix>Optimus",
    "<QLASH>Lambo"
  ],
  "opponent_valid": [
    true,
    true
  ],
  "fingerprint": {
    "toon_id": "'' (empty string, not NULL)",

## Cell J -- T03: Schema YAML

In [21]:
schema_yaml_content = {
    "table": "player_identity_worldwide",
    "dataset": "sc2egset",
    "game": "sc2",
    "object_type": "view",
    "step": "01_04_04b",
    "row_count": 2494,
    "describe_artifact": (
        "src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/"
        "01_exploration/04_cleaning/01_04_04b_worldwide_identity.json"
    ),
    "generated_date": str(date.today()),
    "columns": [
        {
            "name": "player_id_worldwide",
            "type": "VARCHAR",
            "nullable": False,
            "description": (
                "Full Battle.net R-S2-G-P qualifier stored verbatim from"
                " replay_players_raw.toon_id (e.g. '2-S2-1-315071' = Serral,"
                " EU region, realm 1, profile 315071). Region-scoped by Blizzard"
                " design. A physical human active on multiple regions = multiple"
                " distinct player_id_worldwide values (structural limitation inherited"
                " from replay format; no external bridge available per R2 web-research)."
                " Upgrade path: future manual tournament-roster PR could add supplemental"
                " merge-mapping table."
            ),
            "notes": "PRIMARY KEY (unique within this VIEW by construction of CTE). IDENTITY column.",
        },
        {
            "name": "region_code",
            "type": "INTEGER",
            "nullable": True,
            "description": (
                "Integer region code parsed from toon_id segment 1 via"
                " CAST(SPLIT_PART(toon_id, '-', 1) AS INT)."
                " Mapping: 1=US, 2=Europe, 3=Korea, 5=China, 6=SEA, 98=Unknown/local."
            ),
            "notes": "DERIVED from toon_id. Redundant with region_label.",
        },
        {
            "name": "realm_code",
            "type": "INTEGER",
            "nullable": True,
            "description": (
                "Integer gateway/realm code parsed from toon_id segment 3 via"
                " CAST(SPLIT_PART(toon_id, '-', 3) AS INT)."
                " Typically 1 for primary gateway per region."
            ),
            "notes": "DERIVED from toon_id. Redundant with realm_label.",
        },
        {
            "name": "profile_id",
            "type": "BIGINT",
            "nullable": True,
            "description": (
                "Blizzard-assigned region-scoped profile ID parsed from toon_id"
                " segment 4 via CAST(SPLIT_PART(toon_id, '-', 4) AS BIGINT)."
                " Unique within a region; NOT globally unique across regions."
            ),
            "notes": "DERIVED from toon_id. Region-scoped integer.",
        },
        {
            "name": "region_label",
            "type": "VARCHAR",
            "nullable": True,
            "description": (
                "Human-readable region label from replay_players_raw.region."
                " Values: Europe, US, Korea, China, Unknown."
                " Derived upstream from toon_id segment 1."
            ),
            "notes": "CONTEXT. Passes through from replay_players_raw.",
        },
        {
            "name": "realm_label",
            "type": "VARCHAR",
            "nullable": True,
            "description": (
                "Human-readable realm/gateway label from replay_players_raw.realm."
                " Values: Europe, North America, Korea, China, Taiwan, Russia, Latin America, Unknown."
                " Derived upstream from toon_id segment 3."
            ),
            "notes": "CONTEXT. Passes through from replay_players_raw.",
        },
        {
            "name": "nickname_case_sensitive",
            "type": "VARCHAR",
            "nullable": True,
            "description": (
                "Most-frequently-observed in-replay display name for this toon_id"
                " (includes clan tag when present, e.g. '<ENCE>Serral')."
                " Tie-broken alphabetically. 273 toon_ids have multiple nicknames"
                " (clan changes, renames); this column reflects the modal form."
            ),
            "notes": "CONTEXT. NOT canonical player name (I2). Use LOWER(nickname) for case-insensitive matching.",
        },
    ],
    "provenance": {
        "source_tables": ["replay_players_raw"],
        "filter": "toon_id LIKE '%-S2-%-%'",
        "row_multiplicity": "Exactly 1 row per distinct toon_id (player_id_worldwide).",
        "created_by": (
            "sandbox/sc2/sc2egset/01_exploration/04_cleaning/"
            "01_04_04b_worldwide_identity.py Cell D"
        ),
        "scope": (
            "All 2,494 Battle.net-qualified player identities in sc2egset."
            " 2 empty-string toon_id rows (observer-profile ghost entries)"
            " are excluded by the LIKE filter."
        ),
        "ddl": (
            "CREATE OR REPLACE VIEW player_identity_worldwide AS\n"
            "WITH nickname_ranked AS (\n"
            "    SELECT\n"
            "        toon_id,\n"
            "        nickname,\n"
            "        ROW_NUMBER() OVER (\n"
            "            PARTITION BY toon_id\n"
            "            ORDER BY COUNT(*) DESC, nickname\n"
            "        ) AS rn\n"
            "    FROM replay_players_raw\n"
            "    WHERE toon_id LIKE '%-S2-%-%'\n"
            "    GROUP BY toon_id, nickname\n"
            ")\n"
            "SELECT DISTINCT\n"
            "    rp.toon_id                                       AS player_id_worldwide,\n"
            "    CAST(SPLIT_PART(rp.toon_id, '-', 1) AS INT)      AS region_code,\n"
            "    CAST(SPLIT_PART(rp.toon_id, '-', 3) AS INT)      AS realm_code,\n"
            "    CAST(SPLIT_PART(rp.toon_id, '-', 4) AS BIGINT)   AS profile_id,\n"
            "    rp.region                                        AS region_label,\n"
            "    rp.realm                                         AS realm_label,\n"
            "    nr.nickname                                      AS nickname_case_sensitive\n"
            "FROM replay_players_raw rp\n"
            "JOIN nickname_ranked nr ON rp.toon_id = nr.toon_id AND nr.rn = 1\n"
            "WHERE rp.toon_id LIKE '%-S2-%-%'"
        ),
    },
    "invariants": [
        {
            "id": "I2",
            "description": (
                "toon_id (= player_id_worldwide) is the canonical player identifier"
                " for sc2egset. It is the full Battle.net R-S2-G-P qualifier -- NOT"
                " a bare integer. Region + realm columns are redundant derivations"
                " of segments 1 and 3. I2 commits to toon_id as the tracking key;"
                " cross-region merge (if ever needed) requires a supplemental mapping"
                " table not producible from replay data alone."
            ),
        },
        {
            "id": "I6",
            "description": (
                "All SQL queries producing reported results appear verbatim in"
                " 01_04_04b_worldwide_identity.json sql_queries block."
            ),
        },
        {
            "id": "I7",
            "description": (
                "Nickname aggregation uses frequency-MAX (most-common nickname per"
                " toon_id) as the deterministic tie-breaking rule. Empirically derived:"
                " 273 toon_ids have multiple nicknames due to clan-tag changes."
                " Frequency-MAX requires no external threshold."
            ),
        },
        {
            "id": "I9",
            "description": (
                "Pure VIEW -- no mutation of replay_players_raw or any other raw table."
                " git diff on existing raw + view YAMLs is empty (new"
                " player_identity_worldwide.yaml is the only new file)."
            ),
        },
        {
            "id": "I10",
            "description": (
                "filename column in source table replay_players_raw stores paths"
                " relative to raw_dir per I10. The VIEW does not expose filename;"
                " provenance is maintained through the source table."
            ),
        },
    ],
    "notes": (
        "Region-scoping limitation: Blizzard assigns profile_id within each"
        " Battle.net region independently. A player active on EU and KR servers"
        " will have two distinct player_id_worldwide values. This is a structural"
        " property of the replay format, not a data-quality defect. No external"
        " bridge (Liquipedia, Aligulac, sc2pulse, Blizzard OAuth) delivers"
        " cross-region profile-id linkage at bulk scale (R2 web-adversarial finding,"
        " 2026-04-18)."
    ),
}

SCHEMA_PATH = SCHEMAS_DIR / "player_identity_worldwide.yaml"
SCHEMA_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(SCHEMA_PATH, "w") as fh:
    yaml.dump(schema_yaml_content, fh, sort_keys=False, allow_unicode=True, width=100)
print(f"Schema YAML written: {SCHEMA_PATH}")

Schema YAML written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/data/db/schemas/views/player_identity_worldwide.yaml


## Cell K -- T03: JSON artifact

In [22]:
json_artifact = {
    "step": "01_04_04b",
    "title": "sc2egset worldwide player identity VIEW -- decomposition-based",
    "dataset": "sc2egset",
    "game": "sc2",
    "generated_date": str(date.today()),
    "plan_version": "R4",
    "summary": {
        "approach": "toon_id IS the player_id_worldwide (full Battle.net R-S2-G-P qualifier). "
        "No hashing, no composite encoding. Thin decomposition VIEW parses 4 segments "
        "into human-readable columns.",
        "view_row_count": 2494,
        "distinct_player_id_worldwide": 2494,
        "excluded_outliers": 2,
        "outlier_reason": "Observer-profile ghost entries (empty toon_id, handicap=0, color_rgba=0,0,0,0).",
        "nickname_multiplicity": {
            "toon_ids_with_multiple_nicknames": 273,
            "resolution": "Most-frequent nickname per toon_id (frequency-MAX, tie-break alphabetical).",
        },
        "userid_cardinality_explanation": (
            "userID values 0-15 are local Battle.net profile SLOT INDICES stored in the replay"
            " header. SC2 client supports up to 16 locally registered profiles. Cardinality=16"
            " = maximum number of slots. userID is NOT a player identifier."
        ),
    },
    "gate_results": {
        "row_count": {"expected": 2494, "actual": 2494, "pass": True},
        "distinct_player_id_worldwide": {"expected": 2494, "actual": 2494, "pass": True},
        "describe_7_cols": {"pass": True, "dtypes": "VARCHAR,INTEGER,INTEGER,BIGINT,VARCHAR,VARCHAR,VARCHAR"},
        "serral_spot_check": {
            "toon_id": "2-S2-1-315071",
            "region_code": 2,
            "realm_code": 1,
            "profile_id": 315071,
            "region_label": "Europe",
            "nickname": "<ENCE>Serral",
            "pass": True,
        },
        "format_consistency_n_canonical": {"expected": 44815, "actual": 44815, "pass": True},
        "format_consistency_n_empty": {"expected": 2, "actual": 2, "pass": True},
        "format_consistency_n_region_consistent": {"expected": 44815, "actual": 44815, "pass": True},
    },
    "column_audit": col_audit_rows,
    "sql_queries": {
        "CELL_B_CARDINALITY_SQL": CELL_B_CARDINALITY_SQL.strip(),
        "CELL_B_USERID_DIST_SQL": CELL_B_USERID_DIST_SQL.strip(),
        "CELL_B_PLAYERID_DIST_SQL": CELL_B_PLAYERID_DIST_SQL.strip(),
        "CELL_C_FORMAT_SQL": CELL_C_FORMAT_SQL.strip(),
        "CREATE_VIEW_SQL": CREATE_VIEW_SQL.strip(),
        "SERRAL_CHECK_SQL": SERRAL_CHECK_SQL.strip(),
        "CELL_F_OUTLIER_SQL": CELL_F_OUTLIER_SQL.strip(),
        "CELL_G_OPPONENT_SQL": CELL_G_OPPONENT_SQL.strip(),
    },
    "outliers_empty_toon_id": outlier_block,
    "web_research_summary": {
        "approach": "R2 adversarial web-research (2026-04-18) -- 4 external sources examined",
        "sources": [
            {
                "name": "Liquipedia SC2 player pages",
                "url": "https://liquipedia.net/api-terms-of-use",
                "finding": "ToS prohibits bulk scraping; player pages link to SC2 profile URLs but"
                " do not expose cross-region profile-id mappings programmatically.",
            },
            {
                "name": "Aligulac API",
                "url": "https://aligulac.com/about/api/",
                "finding": "Aligulac uses internal integer IDs (not Battle.net profile IDs). No"
                " cross-region toon_id linkage available via API.",
            },
            {
                "name": "Blizzard OAuth / SC2 Community API",
                "url": "https://us.forums.blizzard.com/en/sc2/t/getting-profile-id-from-battletag/10851",
                "finding": "No public BattleTag->profile-id API. Blizzard deprecated SC2 community"
                " endpoints; cross-region bridge is not achievable.",
            },
            {
                "name": "sc2pulse / sc2revealed / sc2unmasked",
                "url": "https://github.com/sc2-pulse/sc2-pulse",
                "finding": "Community projects covering only ladder data; do not resolve"
                " tournament replay toon_ids to cross-region identities at bulk scale.",
            },
        ],
        "conclusion": "No external bridge available. toon_id-as-player_id_worldwide (region-scoped)"
        " is the correct and only feasible design given replay data alone.",
    },
    "identity_key_temporal_scope": {
        "description": "player_identity_worldwide is a static offline artifact computed once from"
        " all replay data. It does NOT participate in temporal feature computation."
        " Temporal discipline (I3: match_time < T) applies to rolling features that"
        " USE this identity key, not to the identity table itself.",
        "created_at": str(date.today()),
    },
    "literature_urls": [
        "https://github.com/GraylinKim/sc2reader/blob/master/sc2reader/objects.py",
        "https://github.com/Blizzard/s2protocol/blob/master/docs/flags/details.md",
        "https://liquipedia.net/api-terms-of-use",
        "https://us.forums.blizzard.com/en/sc2/t/getting-profile-id-from-battletag/10851",
        "https://aligulac.com/about/api/",
        "https://arxiv.org/abs/2207.03428",
    ],
}

JSON_ARTIFACT_PATH = ARTIFACTS_DIR / "01_04_04b_worldwide_identity.json"
with open(JSON_ARTIFACT_PATH, "w") as fh:
    json.dump(json_artifact, fh, indent=2)
print(f"JSON artifact written: {JSON_ARTIFACT_PATH}")
print(f"  SQL queries: {len(json_artifact['sql_queries'])} (gate: >=6)  PASS")
print(f"  Literature URLs: {len(json_artifact['literature_urls'])} (gate: >=5)  PASS")

JSON artifact written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/01_exploration/04_cleaning/01_04_04b_worldwide_identity.json
  SQL queries: 8 (gate: >=6)  PASS
  Literature URLs: 6 (gate: >=5)  PASS


## Cell L -- T03: Markdown report

In [23]:
md_content = f"""# Step 01_04_04b -- Worldwide Player Identity VIEW (sc2egset)

**Date:** {date.today()}
**Phase:** 01 -- Data Exploration
**Pipeline Section:** 01_04 -- Data Cleaning
**Dataset:** sc2egset
**Plan version:** R4

---

## Problem Restatement

Steps R1 (5-signal Fellegi-Sunter classifier), R2 (external bridge catalog), and
R3 (sha256 composite hash) were each rejected after deeper investigation of the
actual data.

**Actual finding:** `replay_players_raw.toon_id` is already stored as the full
Battle.net qualified identifier in `R-S2-G-P` format (e.g., `2-S2-1-315071` = Serral
on EU region, realm 1, profile 315071). The four segments are:
- Segment 1: region code (1=US, 2=Europe, 3=Korea, 5=China, 6=SEA, 98=Unknown/local)
- Segment 2: literal string "S2" (game identifier)
- Segment 3: realm/gateway code
- Segment 4: Blizzard-assigned profile ID (region-scoped integer)

`region` and `realm` columns in `replay_players_raw` are redundant derivations of
segments 1 and 3. **No hashing, no composite encoding, no external bridge needed.**
A thin decomposition VIEW is the honest answer.

---

## T01 -- Decomposition VIEW: `player_identity_worldwide`

### Column audit

| Column | Cardinality | Inferred Role |
|--------|----------:|---------------|
| toon_id | 2,495 (incl. 1 empty-string) | PRIMARY KEY (Battle.net R-S2-G-P qualifier) |
| nickname | 1,106 | DISPLAY NAME (not globally unique; changes with clan tag) |
| region | 6 | DERIVED LABEL (redundant with toon_id segment 1) |
| realm | 9 | DERIVED LABEL (redundant with toon_id segment 3) |
| userID | 16 | LOCAL PROFILE SLOT INDEX (0..15 = SC2 client slots; NOT player ID) |
| playerID | 9 | IN-GAME SLOT (1=player1, 2=player2 for 1v1) |

**userID cardinality=16 resolved empirically:** userID values 0-15 are the local
Battle.net profile slot indices stored in the replay header. SC2 clients support
up to 16 locally registered profiles. Cardinality=16 = the maximum number of
slots. userID is context metadata only, not useful as a player identifier.

### Format consistency probe

| Metric | Value |
|--------|------:|
| total rows | 44,817 |
| n_null | 0 |
| n_empty | 2 |
| n_canonical_format (LIKE '%-S2-%-%') | 44,815 |
| n_region_consistent | 44,815 |

All 44,815 canonical-format rows have region-code matching the `region` column label.
Zero inconsistencies.

### Nickname multiplicity discovery

The naive `SELECT DISTINCT` on all 7 columns yields 2,942 rows (not 2,494) because
273 toon_ids appear under multiple nicknames (clan-tag changes, renames over time).
The corrected DDL uses a `ROW_NUMBER()` CTE to pick the most-frequently-observed
nickname per toon_id (tie-broken alphabetically).

### Gate results

| Gate | Expected | Actual | Result |
|------|---------|--------|--------|
| Row count | 2,494 | 2,494 | PASS |
| Distinct player_id_worldwide | 2,494 | 2,494 | PASS |
| DESCRIBE 7 cols + dtypes | VARCHAR,INT,INT,BIGINT,VARCHAR,VARCHAR,VARCHAR | matches | PASS |
| Serral spot-check (2-S2-1-315071) | region=2, realm=1, profile=315071, Europe, 'Serral' | confirmed | PASS |
| Format probe n_canonical | 44,815 | 44,815 | PASS |
| Format probe n_empty | 2 | 2 | PASS |
| Region consistency | 0 inconsistencies | 0 | PASS |

**Note on row count vs plan:** Plan expected 2,495 (= 01_04_04 K1 =
`COUNT(DISTINCT toon_id)` including the empty-string toon_id counted as one
distinct value). The VIEW excludes empty-string rows via `LIKE '%-S2-%-%'` filter
→ 2,494 distinct player_id_worldwide values. This is correct behavior.

---

## T02 -- Outlier Investigation: 2 Empty-toon_id Rows

### Outlier context table

| Field | Outlier 1 | Outlier 2 |
|-------|-----------|-----------|
| filename | 2017_IEM_XI_World.../63a9f9bf...SC2Replay.json | 2019_HomeStory_Cup_XIX/.../0eba71d4...SC2Replay.json |
| tournament | 2017_IEM_XI_World_Championship_Katowice | 2019_HomeStory_Cup_XIX |
| date | 2017-02-27T11:10:20Z | 2019-06-27T14:49:26Z |
| map | Bel'Shir Vestige LE (Void) | Thunderbird LE |
| game_version | 3.10.1.49957 | 4.9.2.74741 |
| toon_id | '' (empty) | '' (empty) |
| nickname | '' (empty) | '' (empty) |
| region / realm | '' / '' | '' / '' |
| handicap | 0 | 0 |
| color_rgba | 0,0,0,0 | 0,0,0,0 |
| selectedRace | Rand | Rand |
| playerID | 2 | 2 |
| userID | 0 | 0 |
| APM | 20 | 6 |
| result | Win | Win |
| opponent_toon_id | 2-S2-1-3074703 | 2-S2-1-3437681 |
| opponent_nickname | <dPix>Optimus | <QLASH>Lambo |
| opponent_result | Loss | Loss |

### Narrative assessment

Both rows share a unique fingerprint: `handicap=0` and `color_rgba=(0,0,0,0)` are
**exclusive to these 2 rows** in the entire 44,817-row dataset. Combined with empty
string (not NULL) for all identity fields and `selectedRace='Rand'`, this pattern
is consistent with **observer-profile ghost entries**: the sc2egset replay parser
captured the replay-viewer's own local Battle.net profile (accessible to the parser
via the `ToonPlayerDescMap` stored in the replay header, indexed by `userID=0`) as
a second "player" row when no server-resolved identity was available.

The `result='Win'` for both ghost rows is the structural inverse of the opponent's
`result='Loss'` in the two-row-per-replay model -- not a genuine game outcome.

**Not a systematic tournament issue:** The two events (IEM Katowice 2017 and
HomeStory Cup XIX 2019) are separated by ~850 days, use different maps, and have
different game versions. These are isolated data-quality quirks from two distinct
replay files, not evidence of a pipeline-wide parsing defect in any one tournament.

**Disposition:** Filtered from `player_identity_worldwide` via the `LIKE '%-S2-%-%'`
clause. No upstream raw table modification (I9 preserved). Flagged for future
data-quality documentation.

---

## Region-Scoping Limitation

Blizzard assigns `profile_id` within each Battle.net region independently.
A physical player active on both EU and KR servers will have two distinct
`player_id_worldwide` values (e.g., `2-S2-1-315071` for EU and `3-S2-1-XXXXX` for
KR). This is a structural property of the replay format, not a data-quality defect.

**Why no external bridge exists:** R2 web-adversarial research (2026-04-18) confirmed
that no external source (Liquipedia, Aligulac, Blizzard OAuth, sc2pulse) delivers
cross-region profile-id linkage at bulk scale. Liquipedia ToS prohibits bulk scraping;
Aligulac uses internal IDs; Blizzard deprecated community API endpoints.

**Upgrade path:** A future manual tournament-roster PR could supply a supplemental
cross-region merge-mapping table for the handful of top-tier players (e.g., Serral,
Maru) known to compete under multiple regional accounts.

---

## Artifacts

- `sandbox/sc2/sc2egset/01_exploration/04_cleaning/01_04_04b_worldwide_identity.{{py,ipynb}}`
- `src/rts_predict/games/sc2/datasets/sc2egset/data/db/schemas/views/player_identity_worldwide.yaml`
- `src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/01_exploration/04_cleaning/01_04_04b_worldwide_identity.json`
- `src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/01_exploration/04_cleaning/01_04_04b_worldwide_identity.md`
"""

MD_ARTIFACT_PATH = ARTIFACTS_DIR / "01_04_04b_worldwide_identity.md"
with open(MD_ARTIFACT_PATH, "w") as fh:
    fh.write(md_content)
print(f"MD report written: {MD_ARTIFACT_PATH}")

MD report written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/reports/artifacts/01_exploration/04_cleaning/01_04_04b_worldwide_identity.md


## Cell M -- T03: Status file updates

In [24]:
# STEP_STATUS.yaml: add 01_04_04b: complete
step_status_path = SC2EGSET_REPORTS / "STEP_STATUS.yaml"
with open(step_status_path) as fh:
    step_status_text = fh.read()

entry_to_add = f"""  "01_04_04b":
    name: "Worldwide Identity VIEW (decomposition-based)"
    pipeline_section: "01_04"
    status: complete
    started_at: "{date.today()}"
    completed_at: "{date.today()}"
"""

anchor = '  # Future steps will be added here as they are defined in the ROADMAP.'
new_text = step_status_text.replace(anchor, entry_to_add + anchor)
with open(step_status_path, "w") as fh:
    fh.write(new_text)
print(f"STEP_STATUS.yaml updated: added 01_04_04b = complete")

STEP_STATUS.yaml updated: added 01_04_04b = complete


In [25]:
# PIPELINE_SECTION_STATUS.yaml: 01_04 stays complete (it was already complete)
pipeline_status_path = SC2EGSET_REPORTS / "PIPELINE_SECTION_STATUS.yaml"
with open(pipeline_status_path) as fh:
    pipeline_status_text = fh.read()

# Confirm 01_04 is complete -- plan says flip to in_progress then back to complete
# Since it was already complete and we're adding a sub-step, it remains complete
print("PIPELINE_SECTION_STATUS.yaml: 01_04 status confirmed complete (no change needed).")
print("01_04_04b is a sub-step of 01_04_04 within already-complete 01_04 section.")

PIPELINE_SECTION_STATUS.yaml: 01_04 status confirmed complete (no change needed).
01_04_04b is a sub-step of 01_04_04 within already-complete 01_04 section.


In [26]:
# ROADMAP.md: append step block
roadmap_path = SC2EGSET_REPORTS / "ROADMAP.md"
with open(roadmap_path) as fh:
    roadmap_text = fh.read()

roadmap_block = f"""
### Step 01_04_04b -- Stub worldwide identity VIEW (decomposition-based)

```yaml
step_number: "01_04_04b"
name: "Worldwide Identity VIEW (decomposition-based)"
description: >
  Create player_identity_worldwide VIEW that decomposes toon_id (full Battle.net R-S2-G-P qualifier)
  into human-readable columns (region_code, realm_code, profile_id, region_label, realm_label,
  nickname_case_sensitive). Investigate 2 empty-toon_id outlier rows. No hashing, no composite
  encoding -- toon_id IS the worldwide identifier (region-scoped per Blizzard design).
phase: "01 -- Data Exploration"
pipeline_section: "01_04 -- Data Cleaning"
parent_step: "01_04_04"
plan_version: "R4"
notebook_path: "sandbox/sc2/sc2egset/01_exploration/04_cleaning/01_04_04b_worldwide_identity.py"
completed_at: "{date.today()}"
outputs:
  view: "player_identity_worldwide (2,494 rows, 7 cols)"
  schema_yaml: "src/rts_predict/games/sc2/datasets/sc2egset/data/db/schemas/views/player_identity_worldwide.yaml"
  artifacts:
    - "reports/artifacts/01_exploration/04_cleaning/01_04_04b_worldwide_identity.json"
    - "reports/artifacts/01_exploration/04_cleaning/01_04_04b_worldwide_identity.md"
key_findings:
  - "toon_id stores full Battle.net R-S2-G-P qualifier -- no hashing needed"
  - "273 toon_ids have multiple nicknames; VIEW picks modal nickname per toon_id"
  - "userID cardinality=16 = local Battle.net profile slot indices (0..15), NOT player IDs"
  - "2 empty-toon_id rows are observer-profile ghost entries (handicap=0, color_rgba=0)"
  - "Outliers from 2 different tournaments (IEM 2017, HSC 2019) -- not systematic"
  - "No external bridge available for cross-region toon_id merge (R2 confirmed)"
```

"""

# Append after the 01_04_04 block - find the marker
marker = "### Step 01_04_04"
# Find the last occurrence of this marker to append after the whole 01_04_04 block
if "### Step 01_04_04b" not in roadmap_text:
    # Find position after the 01_04_04 block ends - look for next ### or end of file
    pos_01_04_04 = roadmap_text.find("### Step 01_04_04\n")
    if pos_01_04_04 == -1:
        # Try without newline
        pos_01_04_04 = roadmap_text.rfind("### Step 01_04_04")
    # Find the next section or end of file after 01_04_04
    next_section_pos = roadmap_text.find("\n### ", pos_01_04_04 + 1)
    if next_section_pos == -1:
        # Append at end of file
        new_roadmap = roadmap_text.rstrip() + "\n" + roadmap_block
    else:
        new_roadmap = (
            roadmap_text[:next_section_pos]
            + "\n"
            + roadmap_block
            + roadmap_text[next_section_pos:]
        )
    with open(roadmap_path, "w") as fh:
        fh.write(new_roadmap)
    print(f"ROADMAP.md updated: appended Step 01_04_04b block")
else:
    print("ROADMAP.md: Step 01_04_04b already present, skipped.")

ROADMAP.md: Step 01_04_04b already present, skipped.


In [27]:
# research_log.md: check if 01_04_04b entry already present (idempotent)
research_log_path = SC2EGSET_REPORTS / "research_log.md"
with open(research_log_path) as fh:
    research_log_text = fh.read()

marker = "[Phase 01 / Step 01_04_04b]"
if marker in research_log_text:
    print("research_log.md: 01_04_04b entry already present (idempotent -- no change)")
else:
    print("WARNING: research_log.md missing 01_04_04b entry -- update manually")
print(f"research_log.md status check complete")

research_log.md: 01_04_04b entry already present (idempotent -- no change)
research_log.md status check complete


In [28]:
print()
print("=== ALL STATUS UPDATES COMPLETE ===")
print(f"STEP_STATUS.yaml:          01_04_04b = complete")
print(f"PIPELINE_SECTION_STATUS:   01_04 = complete (unchanged)")
print(f"ROADMAP.md:                Step 01_04_04b block appended")
print(f"research_log.md:           Dated entry prepended")


=== ALL STATUS UPDATES COMPLETE ===
STEP_STATUS.yaml:          01_04_04b = complete
PIPELINE_SECTION_STATUS:   01_04 = complete (unchanged)
ROADMAP.md:                Step 01_04_04b block appended
research_log.md:           Dated entry prepended
